In [ ]:
"""
                     Below is the python code for the RAG. 
"""

import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Library for vector database
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# Embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector = embeddings.embed_query("Hello")

# Chat model
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3")

# Parsing the pdf document
pdf_path = r"D:\RAG\RAG_pipeline_1\doc\deepseek-v4-2026.pdf"
pages = PyPDFLoader(pdf_path).load()
print(len(pages))


# chunking the pages
chunker =RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=400)
chunked_data = chunker.split_documents(pages)

index = faiss.IndexFlatL2(384)               # 384 is the dimension of the embedding vector for the "sentence-transform
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)
vector_store.add_documents(chunked_data)     # Adding the chunked data to the vector store

# Retrievel of top 10 relevant context from vector database 
retriever = vector_store.as_retriever(search_kwargs={"k": 10}) #   Retrieving the top 10 relevant documents from the vector store

def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the supplied context.
    If the answer is not available in the context, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

###################################### RAG Chain ############################
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# --------------------------------------------------
# 12. Ask a question related to the PDF
# --------------------------------------------------

answer = rag_chain.invoke(
    "what is trainning framework adopted for  DeepSeek-V4"
)
print(answer)